# pbncalib_botsort — Kaggle pipeline

Option A field registration (PnLCalib + temporal smoothing) inside the SoccerNet
GSR pipeline, with an A/B against the BroadTrack baseline.

**Four stages, each gating the next.** Run them in order and stop at the first
failure — a later stage's output is meaningless if an earlier one did not pass.

| Stage | What | Gates |
|---|---|---|
| 1 | build env (uv + Python 3.9) → `preflight_imports.py` | every stage imports |
| 2 | `verify_pnlcalib_env.py` | **answers the torch question** |
| 3 | shortest sequence end to end | `bbox_pitch` populated and trustworthy |
| 4 | GS-HOTA A/B vs BroadTrack | the actual verdict |

### The environment build is not trivial

Kaggle's session Python is past 3.9 and ships torch 2.x. This pipeline needs
Python 3.9 and torch 1.13.1 — the detector, reid and tracking stages require it.
So Stage 1 installs `uv`, has it fetch a 3.9 interpreter, and builds a separate
venv: the work `preflight_cpu.sh` does elsewhere. **Expect a slow first cell.**

Do **not** `pip install -r optiona_sfr/requirements.txt` into the session. It
would try to move the session's torch and can leave CUDA mismatched against the
driver; the file's own header says so. Everything below installs into the venv
with `--python .venv`, never into the session.

## Stage 1 — environment

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yass1223/pbncalib_botsort.git"
WORK = Path("/kaggle/working")
REPO = WORK / "pbncalib_botsort"
VENV = REPO / ".venv"


def sh(cmd, cwd=None, check=True):
    print("+", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode:
        raise RuntimeError("failed (%d): %s" % (r.returncode, cmd))
    return r.returncode


if not (REPO / "pyproject.toml").exists():
    sh("git clone --depth 1 %s %s" % (REPO_URL, REPO))
else:
    print("repo already present")

In [ ]:
# uv, a Python 3.9 interpreter, and the venv.
sh("pip -q install uv", check=False)
sh("uv python install 3.9")
sh("uv venv --python 3.9 %s" % VENV, cwd=REPO)

# Resolve fresh from pyproject.toml. uv.lock is intentionally absent: shapely,
# matplotlib and scipy became direct dependencies of the calibration stage and
# the old lock predates them, so syncing it would build an environment in which
# the stage fails at import.
sh("uv pip install --python %s -e ." % VENV, cwd=REPO)

PY = "uv run --python %s python" % VENV
sh(PY + " -c \"import torch, numpy; print('venv torch', torch.__version__, "
        "'| numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
# GATE 1: every _target_ declared under configs/ must import.
rc = sh(PY + " scripts/preflight_imports.py", cwd=REPO, check=False)
assert rc == 0, ("Stage 1 failed: some pipeline stages do not import. Fix the "
                 "dependency first -- later stages cannot be interpreted if the "
                 "pipeline is only half-built.")
print("\nStage 1 PASSED")

## Stage 1.5 — read the forks

The `track` / `gta_link` static audit ran against a checkout where `tracklab`,
`bot_sort` and `strong_sort` did **not exist on disk** — they ship inside the
installed tracklab package. Five findings were left UNVERIFIABLE STATICALLY for
exactly that reason. The venv has just materialised them, so settle them now,
before anything expensive runs.

| | Question | Why it matters |
|---|---|---|
| **F2** | Does the engine batch images for an `ImageLevelModule`? | `process()` takes `batch["input"][0]`; `soccernet.yaml` sets `track: {batch_size: 64}`. If honoured, **63 of every 64 frames are silently dropped** |
| **F4** | Does `GMC.apply` log on failure, and return what? | An identity warp means CMC is dead — HOTA 0.6315 vs 0.6687 |
| **F7** | Kalman state `(w,h)` or `(a,h)`? | The classic BoT-SORT/ByteTrack transplant error |
| **F8** | EMA appearance update in `STrack.update`? | Whether appearance genuinely accumulates |
| **F13** | Does the encoder drop NaN `track_id`? | GTA-Link's collision guard creates those deliberately |

**F2 and F4 gate Stage 4.** Both are shared-mode defects: they affect the Option A
and BroadTrack arms identically, so they do *not* bias the A/B — they cost
sensitivity. Either one can depress absolute GS-HOTA far enough to bury a real
calibration difference, which would make a null result unattributable.

In [ ]:
rc = sh(PY + " scripts/probe_forks.py --json /kaggle/working/fork_probe.json",
        cwd=REPO, check=False)

import json
probe = {}
try:
    probe = json.load(open("/kaggle/working/fork_probe.json"))
except Exception as e:
    print("could not read probe summary:", e)

# Record the two that gate Stage 4. Do not stop here -- Stage 3 is still
# informative even if these are bad; it is the A/B that becomes uninterpretable.
F2 = probe.get("F2", {}).get("answer", "UNRESOLVED")
F4 = probe.get("F4", {}).get("answer", "UNRESOLVED")
print("\n" + "=" * 70)
print("Stage 4 gates:  F2 (batching) = %s   |   F4 (GMC logging) = %s" % (F2, F4))
print("=" * 70)
print("Read the F2 evidence lines above and decide explicitly: if batch_size "
      "reaches a DataLoader for image-level modules, STOP and fix it -- every "
      "number after this point would be computed on 1/64 of the frames.")

## Stage 2 — does PnLCalib run on torch 1.13.1?

**This is the gate the whole integration hangs on.** PnLCalib pins torch 2.3.1;
the pipeline pins 1.13.1 and cannot move. The models use only long-stable ops,
but two things cannot be settled by reading source:

- can torch 1.13.1 read a checkpoint archive written by 2.3.1? (check **D**)
- do the `state_dict` keys match the architecture from `hrnetv2_w48.yaml`? (check **E**)

If D or E fail, **do not bump the pipeline's torch.** The remedy — a subprocess
boundary, or re-serialised checkpoints — lands inside the single
`compute_cameras` function in `optiona_api.py`, which exists to contain exactly
this.

In [ ]:
# ~505 MiB of checkpoints. Resume-safe, so re-running is cheap.
sh("bash scripts/setup_pnlcalib.sh", cwd=REPO)

In [ ]:
FRAME = next(iter(sorted(Path("/kaggle/input").rglob("img1/000001.jpg"))), None)
print("frame:", FRAME)
assert FRAME is not None, "no SoccerNet frame found under /kaggle/input"

rc = sh(PY + " scripts/verify_pnlcalib_env.py"
        " --repo pretrained_models/pnlcalib/PnLCalib"
        " --weights-kp pretrained_models/pnlcalib/weights/SV_kp"
        " --weights-line pretrained_models/pnlcalib/weights/SV_lines"
        " --frame %s" % FRAME, cwd=REPO, check=False)
if rc != 0:
    raise SystemExit(
        "Stage 2 FAILED. Read which check failed above.\n"
        "  D (torch.load) -> 1.13.1 cannot read the 2.3.1 archive.\n"
        "  E (state_dict) -> keys/shapes disagree with hrnetv2_w48.yaml.\n"
        "Either way the fix goes inside compute_cameras() in optiona_api.py. "
        "DO NOT bump the pipeline's torch: the detector, reid and tracking "
        "stages require 1.13.1.")
print("\nStage 2 PASSED -- PnLCalib runs on the pipeline's pins")

## Stage 3 — one sequence, end to end

Shortest available sequence, so a failure surfaces in minutes rather than after
a whole split. The two local gates run first: they need no GPU, and the
conversion gate carries the negative control that proves it can fail.

In [ ]:
# GATE 3a: camera-conversion parity + negative control.
rc = sh(PY + " scripts/verify_optiona_conversion.py", cwd=REPO, check=False)
assert rc == 0, "conversion parity failed -- bbox_pitch would be wrong everywhere"

# GATE 3b: engine contract (pure pandas, no torch).
rc = sh(PY + " tests/test_optiona_api_contract.py", cwd=REPO, check=False)
assert rc == 0, "engine contract broken"

In [ ]:
roots = sorted(Path("/kaggle/input").rglob("*/img1"))
assert roots, "no SoccerNet-GSR sequences found under /kaggle/input"
seqs = sorted((len(list(r.glob("*.jpg"))), r.parent) for r in roots)
n_frames, SEQ = seqs[0]
DATA_ROOT = SEQ.parent.parent
print("shortest sequence: %s (%d frames)" % (SEQ.name, n_frames))
print("dataset root:", DATA_ROOT)

In [ ]:
# TEE STDOUT. boxmot's GMC reports a failed motion estimate with print(), not a
# log record (see the F4 probe), so the evidence never reaches the log file. Keep
# the raw stream: it is the primary source for the CMC assertion below.
RUNLOG = WORK / ("run_optiona_%s.txt" % SEQ.name)
sh(PY + " -m tracklab.main -cn soccernet_optiona"
        " dataset.dataset_path=%s dataset.nvid=1"
        " 'dataset.vids_dict.test=[%s]'"
        " experiment_name=optiona_%s 2>&1 | tee %s"
   % (DATA_ROOT, SEQ.name, SEQ.name, RUNLOG), cwd=REPO)

### Tracking assertions

Ordered by consequence, from the `track` / `gta_link` static audit. These are
instruments, not pass/fail gates — except the two that gate Stage 4.

In [ ]:
import re
text = RUNLOG.read_text(errors="replace") if RUNLOG.exists() else ""

print("1. CMC IS ALIVE  (F4 -- gates Stage 4)")
# boxmot prints on a failed estimate; count those against total frames.
warns = re.findall(r"(?i)not enough (?:matching )?points|warning.*gmc|cmc.*fail", text)
print("   GMC failure prints in captured stdout: %d" % len(warns))
print("   frames in sequence: %d" % n_frames)
if len(warns) > 0.05 * n_frames:
    print("   *** CMC IS DEGRADED on >5%% of frames. You are running closer to the")
    print("       CMC-off configuration (HOTA 0.6315) than to 0.6687. Stage 4's")
    print("       absolute numbers are depressed for BOTH arms.")
elif warns:
    print("   some failures, below 5%% -- acceptable")
else:
    print("   no failure prints found. NOTE: absence of prints is not proof CMC is")
    print("   working -- it may simply not print. Confirm from the F4 probe whether")
    print("   the failure path prints at all.")

print("\n2. BATCH SIZE IS 1  (F2 -- gates Stage 4)")
print("   probe verdict: %s" % F2)
print("   (a silently batched tracker would still produce plausible output)")

print("\n3. FEATURE/DETECTION PARITY  (F6 -- now enforced)")
bad = re.findall(r"ReID returned \d+ features for \d+ detections", text)
print("   violations raised: %d  (a RuntimeError now stops the run)" % len(bad))

print("\n4/8. GTA-LINK MERGE HEALTH  (F9, F14)")
for line in re.findall(r"\[GTA-Link\][^\n]*", text):
    print("   " + line.strip()[:150])

print("\n6. ZERO-FEATURE RATE  (F11 -- now logged)")
z = re.findall(r"detections have a ZERO appearance feature", text)
print("   zero-feature warnings: %d  (0 expected on a healthy sequence)" % len(z))

print("\n9. DETECTOR CLASS IDS  (D2)")
# Expected: exactly {0}, and a single entry in model.names. Anything else means
# the cls==0 filter is discarding evaluated roles (goalkeeper / referee).
for line in re.findall(r"\[YOLO-SNFT\] class ids present[^\n]*", text):
    print("   " + line.strip()[:160])
bad_cls = re.findall(r"\[YOLO-SNFT\] checkpoint emits classes[^\n]*", text)
if bad_cls:
    print("   *** MULTI-CLASS CHECKPOINT -- roles are being dropped silently:")
    for line in bad_cls:
        print("       " + line.strip()[:160])
else:
    print("   no multi-class warning -- consistent with a single-class fine-tune")

print("\n10. DETECTOR imgsz  (D3)")
# Expected: matches the yaml's imgsz: 1280.
for line in re.findall(r"\[YOLO-SNFT\] (?:imgsz|checkpoint was TRAINED|checkpoint carries|could not read)[^\n]*", text):
    print("   " + line.strip()[:160])

print("\n11. ReID FEATURE NORMS  (a-path)")
for line in re.findall(r"\[BoT-SORT\] ReID feature norms[^\n]*", text):
    print("   " + line.strip()[:160])
degen = re.findall(r"\[BoT-SORT\] \d+ near-zero and \d+ non-finite[^\n]*", text)
if degen:
    print("   *** DEGENERATE FEATURES -- update_features divides by the norm with")
    print("       no epsilon, so these become nan in the cost matrix:")
    for line in degen:
        print("       " + line.strip()[:160])

### Feature dump for the GTA-Link threshold probe

`docs/GTA_STC2025_PARAM_MAPPING.md` §4 records that STC-2025's
`merge_dist_thres=0.35` is measured with a **different estimator** to our
`appearance_thresh=0.25`: theirs is the mean pairwise cosine distance over all
instance pairs, ours the distance between EMA-averaged embeddings. By Jensen the
former is systematically larger, so the values are not interchangeable.

Settling the conversion needs **real** per-detection features — the gap depends
entirely on real within-tracklet variance, so synthetic data answers the wrong
question. This dumps them.

In [ ]:
# Dump per-detection OSNet features + track_id for the estimator-gap probe.
import gzip
import pickle
import sys

import numpy as np
import torch
from omegaconf import OmegaConf

feat_path = WORK / ("features_%s.npz" % SEQ.name)
try:
    sys.path.insert(0, str(REPO))
    from sn_gamestate.track.gta_link_api import GTALink

    # Re-extract through the SAME code path the merge decision used, so the
    # dumped features are exactly the ones the thresholds act on.
    det = arm_detections("optiona_%s" % SEQ.name)
    assert det is not None and "track_id" in det.columns, "no state with track_id"
    work = det[det["track_id"].notna()].copy()

    cfg = OmegaConf.load(
        REPO / "sn_gamestate/configs/modules/gta_link/gta_link.yaml").cfg
    cfg.osnet_weights = str(next(iter(sorted(
        (REPO / "pretrained_models").rglob("osnet_x1_0_sports.pt"))), ""))
    gl = GTALink(cfg, device="cuda" if torch.cuda.is_available() else "cpu")

    with gzip.open(REPO / "states" / ("optiona_%s.pklz" % SEQ.name), "rb") as f:
        st = pickle.load(f)
    meta = st.get("image_metadatas", st.get("metadatas"))
    feats = gl._extract_features(work, meta)

    np.savez_compressed(
        feat_path,
        feats=feats.astype(np.float32),
        track_id=work["track_id"].to_numpy(),
        image_id=work["image_id"].to_numpy(),
    )
    print("wrote %s  feats=%s  tracklets=%d"
          % (feat_path, feats.shape, work["track_id"].nunique()))
    print("\nProbe to run on this file (pure numpy, no GPU):")
    print("  for every tracklet pair, compute")
    print("    D_pairwise = mean_ij(1 - cos(f_i, g_j))")
    print("    D_ema      = 1 - cos(ema(A), ema(B)),  ema_alpha=0.9")
    print("  regress D_pairwise on D_ema; report slope, intercept, residual spread.")
    print("  Tight residuals -> 0.35 converts to a definite value on our scale.")
    print("  Wide residuals  -> not inter-convertible; tune against GS-HOTA instead.")
except Exception as e:
    print("feature dump skipped (%s: %s)" % (type(e).__name__, e))
    print("The probe needs feats + track_id; re-extract manually if this failed.")

In [ ]:
# 7. Tracklet length distribution + 8. identity counts, from the saved state.
import pickle, gzip
import numpy as np
import pandas as pd

state = REPO / "states" / ("optiona_%s.pklz" % SEQ.name)
if not state.exists():
    state = next(iter(sorted((REPO / "states").glob("*.pklz"))), None)
print("state:", state)

try:
    with gzip.open(state, "rb") as f:
        st = pickle.load(f)
    det = st["detections"] if isinstance(st, dict) and "detections" in st else None
except Exception as e:
    det, st = None, None
    print("could not read state (%s) -- read the run log's eval table instead" % e)

if det is not None and "track_id" in det.columns:
    tid = det["track_id"]
    print("\n7. TRACKLET LENGTH DISTRIBUTION")
    lens = tid.dropna().value_counts()
    print("   identities: %d   median len %.0f   p10 %.0f   p90 %.0f"
          % (len(lens), lens.median(), np.percentile(lens, 10),
             np.percentile(lens, 90)))
    short = int((lens < 20).sum())
    print("   below min_tracklet_len=20: %d/%d (%.0f%%) -- these are EXCLUDED from"
          % (short, len(lens), 100.0 * short / max(len(lens), 1)))
    print("   merging entirely, so a heavy short tail means GTA-Link operated on a")
    print("   small minority of tracks")

    print("\n8b. NON-NULL bbox_pitch  -- FIRST-CLASS RESULT, not a diagnostic")
    # soccernet_game_state.py:91-100 drops any detection with a null track_id,
    # bbox_ltwh OR bbox_pitch (how="any"). A calibration failure therefore does
    # not create false positives -- it removes the detection from scoring
    # entirely. Lost true positives cost recall, silently.
    if "bbox_pitch" in det.columns:
        nn = int(det["bbox_pitch"].notna().sum())
        print("   non-null bbox_pitch: %d/%d (%.1f%%)"
              % (nn, len(det), 100.0 * nn / max(len(det), 1)))
        # A whole sequence nulled by _empty_outputs looks like a clean run.
        for vid, g in det.groupby(det.get("video_id", pd.Series(0, index=det.index))):
            frac = g["bbox_pitch"].notna().mean()
            if frac < 0.10:
                print("   *** COLLAPSE: video %s has %.1f%% non-null bbox_pitch."
                      % (vid, 100.0 * frac))
                print("       _empty_outputs nulls an entire sequence on any early")
                print("       return -- this is a calibration FAILURE presenting as")
                print("       a clean run. Read the [optiona] log lines above.")
    else:
        print("   bbox_pitch column absent -- the calibration stage did not run")

    print("\n8. NaN track_id REACHING THE EVALUATOR  (F13)")
    n_nan = int(tid.isna().sum())
    print("   detections with NaN track_id: %d/%d (%.2f%%)"
          % (n_nan, len(tid), 100.0 * n_nan / max(len(tid), 1)))
    print("   probe verdict on encoder handling: %s"
          % probe.get("F13", {}).get("answer", "UNRESOLVED"))
    if n_nan and probe.get("F13", {}).get("answer") != "DROPS":
        print("   *** these may reach the encoder as spurious unmatched detections")
        print("       and depress GS-HOTA precision in BOTH arms")

### Read the result — completeness is not evidence

Completeness is ~1.0 by construction: smoothing interpolates gaps, carry-forward
fills the rest, and the tracker never returns `None` after lock-on. A camera that
locks on at frame 1, drifts, and is carried for the remaining frames gives zero
errors, 100% completeness and wrong coordinates everywhere. These are the numbers
that separate that case from a working one.

In [ ]:
import json
import numpy as np

calib = sorted((REPO / "optiona_calib").glob("*.json"))
assert calib, "no calibration JSON written -- the stage did not run"
payload = json.loads(calib[0].read_text())
frames = payload["frames"]
names = sorted(frames)

s = np.array([frames[n]["s"] for n in names], float)
fin = s[np.isfinite(s)]
print("sequence %s (%d calibrated frames)\n" % (payload["sequence"], len(names)))
print("s ACROSS THE WHOLE SEQUENCE, not just at lock-on:")
print("  min %.3f   p10 %.3f   median %.3f   p90 %.3f   max %.3f"
      % (fin.min(), np.percentile(fin, 10), np.median(fin),
         np.percentile(fin, 90), fin.max()))
print("  frames below 0.5: %d/%d" % (int((fin < 0.5).sum()), len(fin)))
first = next((n for n in names if np.isfinite(frames[n]["s"])), None)
print("  first lock-on: %s" % first)
print("  NOTE 0.5 is a STARTING HEURISTIC borrowed from BroadTrack's ONLINE")
print("       reinit rule, not a calibrated cut for this post-smoothing score.")

q = len(fin) // 4
if q:
    print("\n  quartile medians: " +
          "  ".join("%.3f" % np.median(fin[i * q:(i + 1) * q]) for i in range(4)))
    print("  a monotonically falling trend = locked on, then drifted")

P = [frames[n]["parameters"] for n in names]
f_ = np.array([p["x_focal_length"] for p in P])
pos = np.array([p["position_meters"] for p in P])
print("\ncamera parameters:")
print("  focal  %8.1f .. %8.1f   (spread %.1f%% of mean)"
      % (f_.min(), f_.max(), (f_.max() - f_.min()) / f_.mean() * 100))
for i, ax in enumerate("xyz"):
    print("  pos %s  %8.2f .. %8.2f m" % (ax, pos[:, i].min(), pos[:, i].max()))
travel = float(np.sum(np.linalg.norm(np.diff(pos, axis=0), axis=1)))
print("  focal-point travel: %.1f m" % travel)
print("  (NBJW travels up to 20 m per sequence; large travel means the")
print("   focal/distance degeneracy is unchecked)")

In [ ]:
# Out-of-bounds count, straight from the stage's own log line.
# LOWER BOUND, not a measurement: a behind-camera unprojection can land inside
# the bounds by coincidence, so 0% means "none caught", not "none present".
sh("grep -h 'projected positions' outputs/optiona_%s/*/*/*.log 2>/dev/null "
   "|| grep -rh 'projected positions' outputs/ 2>/dev/null | tail -5"
   % SEQ.name, cwd=REPO, check=False)

## Stage 4 — GS-HOTA A/B vs BroadTrack

The only number that settles whether Option A is better. Every other stage is
frozen, so the delta is attributable to calibration alone.

In [ ]:
# GATE: F2 and F4 decide whether Stage 4 can be interpreted.
#
# Neither biases the A/B -- both defects are shared by the Option A and BroadTrack
# arms identically. What they cost is SENSITIVITY: a tracker running on 1/64 of
# the frames, or with camera-motion compensation dead, depresses absolute GS-HOTA
# far enough that a real calibration difference can hide inside the noise. A null
# result under either condition is unattributable, not evidence of equivalence.
blocked = []
if F2 not in ("READ",) or "batch" in str(F2).lower():
    pass  # F2's verdict is READ + evidence; the human decision is recorded below
if F4.startswith("PRINTS") or F4 in ("SILENT", "UNRESOLVED"):
    blocked.append("F4=%s (GMC failures may be invisible)" % F4)

F2_OK = True   # <-- set False if the Stage 1.5 evidence showed real batching
F4_OK = len(warns) <= 0.05 * n_frames

print("F2_OK =", F2_OK, " F4_OK =", F4_OK)
if not (F2_OK and F4_OK):
    print("\n*** Stage 4 will still RUN, but its verdict is NOT usable:")
    for b in blocked:
        print("      -", b)
    print("    Fix the shared-mode defect and re-run before reading any A/B delta.")

In [ ]:
bt_state = REPO / "states" / ("broadtrack_%s.pklz" % SEQ.name)
if not bt_state.exists():
    sh("bash scripts/setup_broadtrack.sh", cwd=REPO, check=False)
    sh(PY + " -m tracklab.main -cn soccernet"
            " dataset.dataset_path=%s dataset.nvid=1"
            " 'dataset.vids_dict.test=[%s]'"
            " experiment_name=broadtrack_%s" % (DATA_ROOT, SEQ.name, SEQ.name),
       cwd=REPO, check=False)
else:
    print("using cached BroadTrack run:", bt_state)

### bbox_pitch accounting — read this before the HOTA table

`soccernet_game_state.py:91-100` drops any detection whose `track_id`,
`bbox_ltwh` **or** `bbox_pitch` is null (`how="any"`). Calibration failure
therefore does not generate false positives — it removes detections from scoring.

So if the two arms retain different numbers of detections, **the GS-HOTA delta
partly measures willingness to abstain rather than calibration quality.** When
that happens, both arms are also scored on the intersection, which is the only
comparison that isolates calibration.

In [ ]:
import gzip
import pickle

import numpy as np
import pandas as pd


def arm_detections(tag):
    p = REPO / "states" / ("%s.pklz" % tag)
    if not p.exists():
        return None
    try:
        with gzip.open(p, "rb") as f:
            st = pickle.load(f)
        return st["detections"] if isinstance(st, dict) and "detections" in st else None
    except Exception as e:
        print("  could not read %s: %s" % (p.name, e))
        return None


A = arm_detections("optiona_%s" % SEQ.name)
B = arm_detections("broadtrack_%s" % SEQ.name)

SCORED = {}
for name, d in (("optiona", A), ("broadtrack", B)):
    if d is None or "bbox_pitch" not in d.columns:
        print("%-11s : no state / no bbox_pitch column" % name)
        continue
    keep = d["bbox_pitch"].notna()
    for col in ("track_id", "bbox_ltwh"):
        if col in d.columns:
            keep &= d[col].notna()
    SCORED[name] = set(d.index[keep])
    print("%-11s : %d/%d detections survive the evaluator's dropna (%.1f%%)"
          % (name, keep.sum(), len(d), 100.0 * keep.mean()))

if len(SCORED) == 2:
    a, b = SCORED["optiona"], SCORED["broadtrack"]
    inter = a & b
    union = a | b
    diff = abs(len(a) - len(b)) / max(len(union), 1)
    print("\nintersection: %d   symmetric difference: %d   imbalance: %.1f%%"
          % (len(inter), len(union - inter), 100.0 * diff))
    if diff > 0.02:
        print("\n*** ARMS ARE NOT SCORED ON THE SAME DETECTIONS (>2%% imbalance).")
        print("    The headline HOTA delta below conflates calibration quality with")
        print("    abstention rate. Score both arms on the %d shared detections and" % len(inter))
        print("    report THAT as the calibration result:")
        print("      tracklab ... eval.eval_set=test  # with detections restricted")
        print("      to the intersection index, or post-filter both states to")
        print("      `inter` and re-run the evaluator on each.")
        pd.Series(sorted(inter)).to_csv(WORK / "scored_intersection.csv", index=False,
                                        header=["detection_index"])
        print("    intersection index written to %s" % (WORK / "scored_intersection.csv"))
    else:
        print("\narms are balanced within 2%% -- the HOTA delta below is attributable")
        print("to calibration rather than to abstention.")

In [ ]:
import re

for tag in ("optiona_%s" % SEQ.name, "broadtrack_%s" % SEQ.name):
    d = REPO / "outputs" / tag
    print("\n=== %s ===" % tag)
    if not d.exists():
        print("  no output directory -- did the run complete?")
        continue
    found = False
    for h in sorted(d.rglob("*.json"))[-5:]:
        try:
            obj = json.loads(h.read_text())
        except Exception:
            continue
        if not isinstance(obj, dict):
            continue
        for k, v in obj.items():
            if re.search(r"hota|deta|assa|accuracy", str(k), re.I):
                print("  %s: %s" % (k, v))
                found = True
    if not found:
        print("  no HOTA-like keys found; check the run log for the eval table")

## What "ready" means

Not "it ran". Ready is:

- **parity + negative control** from Stage 3a, with the control failing loudly
- **`s` across the whole sequence**, with no falling quartile trend
- **out-of-bounds count** at or near zero — remembering it is a lower bound
- **camera parameters physically plausible** — focal not drifting, optical
  centre not wandering off the touchline
- **non-null `bbox_pitch` per arm**, balanced within 2% — or the A/B scored on
  the intersection instead
- **GS-HOTA at least matching BroadTrack** on identical sequences

If any is missing, the run is not evidence — regardless of completeness.

See `KNOWN_LIMITATIONS.md` for the four defects that are understood and
deliberately not fixed, and what each one does and does not invalidate.